# nb_02_features - the gold layer: features, label, star schema

One notebook, two modes:

| `mode` | Input | Output |
|---|---|---|
| `training` | `silver_orders` + `silver_shipments` (label) + reference tables | `gold_training_delivery`, `gold_fact_orders`, `gold_dim_carrier`, `gold_dim_region`, `gold_dim_warehouse`, `gold_dim_date` |
| `open_orders` | `bronze_open_orders` (copied nightly from `sqldb_brightcart_ops`) | `gold_open_orders_features` (no label) |

The feature code is pandas on the driver: the data is small (~90k rows) and the **exact same function** is used for training, for the
nightly batch and in the generator's tests, so the three can never drift apart.

## The leakage rule (kit 02 §4.5)

Only use what is **known at checkout**:

- **Allowed:** order attributes (carrier, warehouse, region, items, weight, value, ship method, promised days, time of day),
  the warehouse's `backlog_index` from the **previous day**, and the weather **forecast** issued the day before.
- **Not allowed:** `shipped_ts`, `delivered_ts`, `actual_severity`, and anything recorded after `order_ts`.

Shipments are joined for the **label** (`is_late`) only. If a feature would not exist at the moment the customer clicks "Buy", it can't be in the model.

**Parameters.** `mode` is set by the nightly pipeline (`open_orders`); run interactively with `training` for Lab 2.

In [ ]:
mode = "training"            # "training" | "open_orders"
holdout_from = "2026-07-01"  # orders on/after this date are the holdout (never trained on)

**Imports and the contract.** The 16 input columns, in order, and which are long vs double.

In [ ]:
import pandas as pd, numpy as np
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
assert mode in ("training", "open_orders"), f"unknown mode {mode!r}"

# The 16 model inputs, in contract order (kit/contracts/feature_contract.json). Order matters: requests use orientation "values".
FEATURES = ["carrier_id", "ship_method_express", "warehouse_id", "region_id", "remoteness", "distance_km", "order_hour",
            "order_dow", "is_weekend", "is_peak", "items_count", "total_weight_kg", "order_value", "promised_days",
            "backlog_index", "forecast_severity"]
FLOAT_COLS = ["distance_km", "total_weight_kg", "order_value", "backlog_index"]   # double
INT_COLS = [c for c in FEATURES if c not in FLOAT_COLS]                           # long

**Helpers to read lakehouse tables into pandas** with dates normalised to text, so joins on dates behave identically whatever the source type.

In [ ]:
def as_day(s):
    """Date-like column -> 'YYYY-MM-DD' strings, so joins on dates behave the same whatever type Spark hands over."""
    return pd.to_datetime(s).dt.strftime("%Y-%m-%d")

def table_to_pandas(name, date_cols=()):
    """Reads a lakehouse table into pandas (our tables are small: at most ~90k rows)."""
    pdf = spark.read.table(name).toPandas()
    for c in date_cols:
        pdf[c] = as_day(pdf[c])
    return pdf

**The feature function.** Mirrors `data/generator/features.py` line for line, including the distance jitter and the fill-in defaults.

In [ ]:
def build_features(orders, regions, calendar, daily_ops, weather) -> pd.DataFrame:
    """Derives the 16 contract features from what is known at checkout. Same logic as data/generator/features.py."""
    o = orders.drop_duplicates("order_id").copy()
    ts = pd.to_datetime(o.order_ts, utc=True)
    o["date"] = ts.dt.strftime("%Y-%m-%d")
    o["order_hour"] = ts.dt.hour
    o["order_dow"] = ts.dt.day_of_week
    o["ship_method_express"] = (o.ship_method == "express").astype(int)
    o["promised_days"] = (pd.to_datetime(o.promised_date) - ts.dt.tz_localize(None).dt.normalize()).dt.days
    o = o.merge(regions[["region_id", "remoteness", "base_distance_km"]], on="region_id", how="left")
    # distance_km is not in the raw orders: a deterministic jitter of the region's base distance, keyed on order_id
    jitter = 0.8 + 0.4 * ((o.order_id * 2654435761) % 1000) / 1000
    o["distance_km"] = (o.base_distance_km * jitter).round(1)
    o = o.merge(calendar[["date", "is_weekend", "is_peak"]], on="date", how="left")
    # backlog of the SAME warehouse on the PREVIOUS day (known at checkout). Missing -> 0.8 (a normal day)
    prev = (pd.to_datetime(o.date) - pd.Timedelta(days=1)).dt.strftime("%Y-%m-%d")
    ops = daily_ops.rename(columns={"date": "prev_date"})[["prev_date", "warehouse_id", "backlog_index"]]
    o["prev_date"] = prev
    o = o.merge(ops, on=["prev_date", "warehouse_id"], how="left")
    o["backlog_index"] = o.backlog_index.fillna(0.8)
    # weather FORECAST for the order's region and day (issued the day before). Missing -> 0 (calm)
    o = o.merge(weather[["date", "region_id", "forecast_severity"]], on=["date", "region_id"], how="left")
    o["forecast_severity"] = o.forecast_severity.fillna(0).astype(int)
    for c in INT_COLS: o[c] = o[c].astype("int64")
    for c in FLOAT_COLS: o[c] = o[c].astype("float64")
    return o

**Load the reference tables** (OneLake shortcuts to `lh_reference`, or the tables `nb_00_bootstrap` wrote).

In [ ]:
regions = table_to_pandas("ref_regions")
calendar = table_to_pandas("ref_calendar", date_cols=["date"])
carriers = table_to_pandas("ref_carriers")
warehouses = table_to_pandas("ref_warehouses")
daily_ops = table_to_pandas("silver_daily_ops", date_cols=["date"])
weather = table_to_pandas("silver_weather", date_cols=["date"])
print(len(regions), "regions,", len(calendar), "calendar days,", len(daily_ops), "ops rows,", len(weather), "weather rows")

**Pick the orders for this mode.** Training uses the cleaned history; open orders come from the operational copy (same columns plus `status`).

In [ ]:
source = "silver_orders" if mode == "training" else "bronze_open_orders"
orders = table_to_pandas(source)
orders["order_ts"] = pd.to_datetime(orders["order_ts"], utc=True)
orders["promised_date"] = as_day(orders["promised_date"])
print(f"{mode}: {len(orders)} orders from {source}")

**Build the features** (both modes). Nulls in feature columns are not allowed; the function fills the two join-dependent ones.

In [ ]:
feat = build_features(orders, regions, calendar, daily_ops, weather)
assert feat[FEATURES].isna().sum().sum() == 0, "feature columns must not contain nulls"
print(feat[FEATURES].dtypes.to_string())

**Write a pandas frame as a Delta table**, casting to the Spark types the contract promises (long / double).

In [ ]:
def save_gold(pdf, table):
    sdf = spark.createDataFrame(pdf)
    for c in INT_COLS:
        if c in pdf.columns: sdf = sdf.withColumn(c, F.col(c).cast("long"))
    for c in FLOAT_COLS:
        if c in pdf.columns: sdf = sdf.withColumn(c, F.col(c).cast("double"))
    sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"wrote {table}: {len(pdf)} rows, {len(pdf.columns)} columns")

**open_orders mode: write `gold_open_orders_features`** and stop. No label exists yet - these orders are still in transit.

In [ ]:
if mode == "open_orders":
    keep = ["order_id", "order_value", "ship_method", "promised_date", "order_ts"] + [c for c in FEATURES if c != "order_value"]
    out = feat[keep].copy()
    out["promised_date"] = pd.to_datetime(out["promised_date"]).dt.date
    save_gold(out, "gold_open_orders_features")
    notebookutils.notebook.exit("gold_open_orders_features written")

**training mode: add the label and the split.** `is_late` = delivered after the promised date. Everything from `holdout_from` onward is the holdout.

In [ ]:
ship = table_to_pandas("silver_shipments")
feat = feat.merge(ship[["order_id", "delivered_ts"]], on="order_id", how="inner")
delivered_day = pd.to_datetime(feat.delivered_ts, utc=True).dt.tz_localize(None).dt.normalize()   # tz-safe: works for naive or 'Z' timestamps
feat["is_late"] = (delivered_day > pd.to_datetime(feat.promised_date)).astype(int)
feat["split"] = np.where(feat.order_ts < pd.Timestamp(holdout_from, tz="UTC"), "train", "holdout")
print(feat["split"].value_counts().to_string())

**`gold_training_delivery`**: the 16 features in contract order, then the label, the split and two keys for tracing rows back.

In [ ]:
training = feat[FEATURES + ["is_late", "split", "order_id", "order_ts"]].copy()
training["order_ts"] = training["order_ts"].dt.tz_localize(None)   # Spark timestamp (session tz is UTC)
save_gold(training, "gold_training_delivery")

**`gold_fact_orders`**: order grain for Power BI, with keys to the four dimensions.

In [ ]:
fact = feat[["order_id", "order_ts", "customer_id", "carrier_id", "region_id", "warehouse_id", "ship_method", "channel",
             "items_count", "total_weight_kg", "order_value", "promised_days", "is_peak", "is_late", "split"]].copy()
fact["date_key"] = fact["order_ts"].dt.strftime("%Y%m%d").astype("int64")
fact["order_ts"] = fact["order_ts"].dt.tz_localize(None)
save_gold(fact, "gold_fact_orders")

**Dimensions.** Carrier, region (with a readable remoteness label) and warehouse come from the reference tables; the date dimension from `ref_calendar`.

In [ ]:
remoteness_label = {0: "Urban", 1: "Suburban", 2: "Rural", 3: "Remote"}
dim_region = regions.copy()
dim_region["remoteness_label"] = dim_region["remoteness"].map(remoteness_label)
dim_region["remoteness"] = dim_region["remoteness"].astype("int64")

dim_date = calendar.copy()
d = pd.to_datetime(dim_date["date"])
dim_date["date_key"] = d.dt.strftime("%Y%m%d").astype("int64")
dim_date["date"] = d.dt.date
dim_date["year"] = d.dt.year.astype("int64")
dim_date["month"] = d.dt.month.astype("int64")
dim_date["month_name"] = d.dt.strftime("%b")
dim_date["year_month"] = d.dt.strftime("%Y-%m")
dim_date["period"] = np.where(dim_date["is_peak"] == 1, "Peak", "Off-peak")
dim_date = dim_date[["date_key", "date", "year", "month", "month_name", "year_month", "dow", "is_weekend", "is_holiday",
                     "is_peak", "period", "holiday_name", "promo_event"]]

save_gold(carriers, "gold_dim_carrier")
save_gold(dim_region, "gold_dim_region")
save_gold(warehouses, "gold_dim_warehouse")
save_gold(dim_date, "gold_dim_date")

**Check cell (Lab 2 checkpoint).** Expect ~90k rows, a late rate near 20%, and carrier 3 in rural/remote regions late ≥ 35% of the time.

In [ ]:
g = spark.read.table("gold_training_delivery")
n = g.count()
late = g.agg(F.avg("is_late")).collect()[0][0]
cell = g.filter((F.col("carrier_id") == 3) & (F.col("remoteness") >= 2)).agg(F.avg("is_late"), F.count("*")).collect()[0]
print(f"rows: {n}")
print(f"late rate overall: {late:.1%}   (target 18-22%)")
print(f"carrier 3, remoteness >= 2: {cell[0]:.1%} late over {cell[1]} orders   (target >= 35%)")
print("columns:", len(g.columns), "->", g.columns[:16] == FEATURES and "contract order OK" or "CONTRACT ORDER BROKEN")

**Late rate by carrier × remoteness** - the same table Lab 2 shows in T-SQL from the SQL analytics endpoint.

In [ ]:
(g.groupBy("carrier_id", "remoteness").agg(F.round(F.avg("is_late"), 3).alias("late_rate"), F.count("*").alias("orders"))
  .orderBy("carrier_id", "remoteness").show())